In [1]:
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax
import math
import copy
import spacy
from datasets import load_dataset
from torch.utils.data import DataLoader
import torch
from torch import tensor
from torch.nn.functional import pad

### Config

In [2]:
d_model=512
h=8
d_ff=2048
dropout=0.1
N=6

src_vocab, tgt_vocab = 11010, 19621
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Decoder Layer

### Model

In [3]:
from networks import SublayerConnection, MultiHeadedAttention, PositionwiseFeedForward, clones, subsequent_mask, \
    Encoder, EncoderLayer, Embeddings, PositionalEncoding

In [4]:
class DecoderLayer(nn.Module):
    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super(DecoderLayer, self).__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 3)

    def forward(self, x, memory, src_mask, tgt_mask):
        m = memory
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        return self.sublayer[2](x, self.feed_forward)

### Inference

#### Src Embedding

In [15]:
src_input = tensor([[
    0, 1101, 7165, 1913, 2433, 7559, 4354, 5914, 10814, 10805,
    10019, 4389, 9040, 2588, 10022, 20, 1, 2, 2, 2,
    2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
    2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
    2, 2, 2, 2, 2, 2, 2, 2, 2, 2
]])
pad=2
src_mask = (src_input != pad)

In [21]:
src_mask[0][:18]

tensor([ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True, False])

In [22]:
src_input[0][:18]

tensor([    0,  1101,  7165,  1913,  2433,  7559,  4354,  5914, 10814, 10805,
        10019,  4389,  9040,  2588, 10022,    20,     1,     2])

In [23]:
position_encoding = PositionalEncoding(d_model, dropout)
src_embed = nn.Sequential(Embeddings(d_model, src_vocab), copy.deepcopy(position_encoding))

In [26]:
src_embed_result = src_embed(src_input)

#### Encoder

In [25]:
self_attn = MultiHeadedAttention(h, d_model)
feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)

encoder_layer = EncoderLayer(d_model, self_attn, feed_forward, dropout)

In [27]:
memory = encoder_layer(src_embed_result, mask=src_mask)

In [28]:
memory.shape

torch.Size([1, 50, 512])

### Decoder Layer